In [ ]:
import joblib
import pandas as pd
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier # Crucial for multi-label
from sklearn.metrics import classification_report, f1_score, make_scorer
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize tools
nltk.download('stopwords')
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Some constants
prefix = "test_"
model_path = "../resources/models"
logistic_regression_pkl = f"{prefix}logistic_regression.pkl"
naive_bayes_pkl = f"{prefix}naive_bayes.pkl"
vectorizer_pkl = f"{prefix}text_vectorizer.pkl"
dataset = 'synonym_youtoxic_english_1000.csv'
file_path = f"../resources/dataset/{dataset}"

df = pd.read_csv(file_path)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization (reducing words to their base or root form)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into a single string
    return ' '.join(tokens)


###########################################################################################

# 1. Clean Text data
print("Cleaning text...")
df['Cleaned_Text'] = df['Text'].apply(preprocess_text)



# 2. Prepare X and y
df_work = df.drop_duplicates(subset=['Text'])
X = df_work['Cleaned_Text']

target_cols = [
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist',
    'IsThreat',
    'IsReligiousHate',
    'IsNationalist'
]
y = df_work[target_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# tfidf_vectorizer = TfidfVectorizer(max_features=10000, analyzer='char', ngram_range=(1, 2))
tfidf_vectorizer = TfidfVectorizer(max_features=10000, min_df=10, stop_words='english', analyzer='char', ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

micro_f1_scorer = make_scorer(f1_score, average='micro')

# 3. Train logistic regression
print("\nTraining Logistic Regression...")
micro_f1_scorer = make_scorer(f1_score, average='micro')

param_grid = {
    # The 'estimator__C' targets the C parameter of the base LogisticRegression estimator
    'estimator__C': [1.0, 0.5, 0.1, 0.05, 0.01], 
    # Optional: Test different regularization penalties (L1 for feature selection, L2 for general)
    'estimator__penalty': ['l2'] 
}

base_lr = LogisticRegression(
    solver='liblinear',
    random_state=42,
    # Crucial for stability in imbalanced data:
    class_weight='balanced', 
    max_iter=50000 
)
moc = MultiOutputClassifier(base_lr, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=moc,
    param_grid=param_grid,
    scoring=micro_f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("Optimizing...")
grid_search.fit(X_train_tfidf, y_train)
print(f"\nBest Micro F1 Score: {grid_search.best_score_:.4f}")
print(f"Best Parameters found: {grid_search.best_params_}")

base_lr_final = LogisticRegression(
    C=grid_search.best_params_['estimator__C'],
    penalty=grid_search.best_params_['estimator__penalty'],
    solver='liblinear',        # Using liblinear/saga (adjust based on your best solver)
    random_state=42,
    max_iter=50000,
    class_weight='balanced'    # Recommended to keep this for imbalance compensation
)

final_model_logistic_regression = MultiOutputClassifier(base_lr_final, n_jobs=-1)
final_model_logistic_regression.fit(X_train_tfidf, y_train)
y_pred_final = final_model_logistic_regression.predict(X_test_tfidf)

report = classification_report(
    y_test, 
    y_pred_final, 
    target_names=target_cols, 
    zero_division=0
)

print("Classification Report:")
print(report)

# Micro-Averaged F1-Score
micro_f1 = f1_score(y_test, y_pred_final, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance): {micro_f1:.4f}")

# 4. Train naive bayes
print("\nTraining Naïve Bayes...")
micro_f1_scorer = make_scorer(f1_score, average='micro', zero_division=0)

param_grid_mnb = {
    'estimator__alpha': [1.0, 0.5, 0.1, 0.01] 
}
base_mnb = ComplementNB()
multi_target_mnb = MultiOutputClassifier(base_mnb, n_jobs=-1)

mnb_grid_search = GridSearchCV(
    estimator=multi_target_mnb,
    param_grid=param_grid_mnb,
    scoring=micro_f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("Optimizing...")
mnb_grid_search.fit(X_train_tfidf, y_train) 
print(f"\nBest Micro F1 Score: {grid_search.best_score_:.4f}")
print(f"Best MNB Alpha: {mnb_grid_search.best_params_}")

base_mnb = ComplementNB(alpha=mnb_grid_search.best_params_['estimator__alpha'])
final_model_naive_bayes = MultiOutputClassifier(base_mnb, n_jobs=-1)
final_model_naive_bayes.fit(X_train_tfidf, y_train)
y_pred_final = final_model_naive_bayes.predict(X_test_tfidf)

report = classification_report(
    y_test, 
    y_pred_final, 
    target_names=target_cols, 
    zero_division=0
)

print("Classification Report:")
print(report)

# Micro-Averaged F1-Score
micro_f1 = f1_score(y_test, y_pred_final, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance): {micro_f1:.4f}")

# 5. Calculate overfitting
y_pred_train = final_model_logistic_regression.predict(X_train_tfidf)
y_pred_test = final_model_logistic_regression.predict(X_test_tfidf)

f1_score_train = f1_score(y_train, y_pred_train, average='micro')
f1_score_test = f1_score(y_test, y_pred_test, average='micro')

print(f"\nOverfitting for Logistic Regression Model: {f1_score_train-f1_score_test:4f}")

y_pred_train = final_model_naive_bayes.predict(X_train_tfidf)
y_pred_test = final_model_naive_bayes.predict(X_test_tfidf)

f1_score_train = f1_score(y_train, y_pred_train, average='micro')
f1_score_test = f1_score(y_test, y_pred_test, average='micro')

print(f"Overfitting for Naïve Bayes Model: {f1_score_train-f1_score_test:4f}")


# 6. Save models
joblib.dump(final_model_logistic_regression, f"{model_path}/{logistic_regression_pkl}")
joblib.dump(final_model_naive_bayes, f"{model_path}/{naive_bayes_pkl}")
joblib.dump(tfidf_vectorizer, f"{model_path}/{vectorizer_pkl}")

print("\nModels trained!")


[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Cleaning text...

Training Logistic Regression...
Optimizing...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Best Micro F1 Score: 0.6700
Best Parameters found: {'estimator__C': 1.0, 'estimator__penalty': 'l2'}
Classification Report:
                 precision    recall  f1-score   support

        IsToxic       0.79      0.76      0.77       276
      IsAbusive       0.70      0.73      0.71       213
  IsProvocative       0.54      0.74      0.63        96
      IsObscene       0.42      0.83      0.56        54
   IsHatespeech       0.53      0.79      0.64        82
       IsRacist       0.53      0.81      0.64        78
       IsThreat       0.50      1.00      0.67        10
IsReligiousHate       0.33      0.67      0.44         3
  IsNationalist       0.22      1.00      0.36         4

      micro avg       0.62      0.77      0.68       816
      macro avg       0.51      0.81      0.60       816
   weighted avg       0.65      0.77      0.69       816
    samp